In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import librosa
# import librosa.display
import IPython.display as ipd
# import sklearn
from sklearn.svm import SVC
# import sys
from glob import glob
from itertools import cycle
import seaborn as sns
from pathlib import Path
from collections import Counter
from sklearn.preprocessing import LabelEncoder
# from huggingface_hub import snapshot_download
# from huggingface_hub import login
# from datasets import load_dataset

In [2]:
def cleanUp(audioInfo):
   X = [item["features"] for item in audioInfo]
   Y = [item["chord"] for item in audioInfo]
   X = np.array(X)
   Y = np.array(Y)
   # print(X.shape)
   # print(Y.shape)
   # print(Counter(Y))
   return X, Y


In [3]:
def create_dataset(audioFiles):
    audioInfo = []
    target_duration = 5
    for file in audioFiles:
        y, sr = librosa.load(file)
        if(len(y) > target_duration * sr):
            y_fixed = y[:target_duration * sr]
        else:
            padding = (target_duration * sr) - len(y)
            y_fixed = np.pad(y, (0, padding), mode="constant")
        S = librosa.feature.melspectrogram(y=y_fixed, sr=sr, n_mels=128)
        S_db_mel = librosa.amplitude_to_db(S, ref=np.max)
        chordName = Path(file).parent.name
        audioInfo.append({"features": S_db_mel, "chord": chordName})
    return cleanUp(audioInfo)

In [4]:
trainFiles = glob("./raw_dataset/Train/*/*.wav")
testFiles = glob("./raw_dataset/Test/*/*.wav")
X_train, Y_train = create_dataset(trainFiles)
X_test, Y_test = create_dataset(testFiles)

In [5]:
import os
print("Current working directory:", os.getcwd())
print("Training files found:", len(trainFiles))
print("Test files found:", len(testFiles))

Current working directory: /mnt/c/Users/oyin1/OneDrive - UT Arlington/Documents/Chord Recognition
Training files found: 633
Test files found: 147


In [6]:
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(Y_train)
y_test_encoded = encoder.transform(Y_test)

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# print(X_train_flat.shape)
# print(X_test_flat.shape)


# print(encoder.classes_)
# print(y_train_encoded[350:1000])
# print(y_test_encoded[350:1000])

Train SVM Model

In [7]:
model = SVC(kernel = "rbf", probability=True)
model.fit(X_train_flat, y_train_encoded)
accuracy = model.score(X_test_flat, y_test_encoded)
print(f"Accuracy: {accuracy:.2f}")

/mnt/c/Users/oyin1/OneDrive - UT Arlington/Documents/Chord Recognition/wsl-env/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Accuracy: 0.81


My Guitar Plays

In [8]:
def testAudio(file):
    target_duaration = 5
    y,sr = librosa.load(file)
    target_samples = target_duaration * sr
    if len(y) > target_samples:
        y_fixed = y[:target_samples]
    else:
        padding = target_samples - len(y)
        y_fixed = np.pad(y, (0, padding), mode="constant")
    S = librosa.feature.melspectrogram(y=y_fixed, sr=sr, n_mels=128)
    S_db_mel = librosa.amplitude_to_db(S, ref=np.max)
    features = S_db_mel.reshape(1, -1)

    probs = model.predict_proba(features)[0]
    best_idx = np.argmax(probs)
    chord = encoder.classes_[best_idx]
    confidence = probs[best_idx]
    return chord, confidence

In [9]:
myPlays = glob("./MyPlays/*.wav")
# print(myPlays)
print(f"Chord Played: {testAudio(myPlays[0])}") #A
print(f"Chord Played: {testAudio(myPlays[1])}") #Em
print(f"Chord Played: {testAudio(myPlays[2])}") #G
print(f"Chord Played: {testAudio(myPlays[3])}") #Em  not so good audio

Chord Played: (np.str_('Am'), np.float64(0.2815522076982415))
Chord Played: (np.str_('Am'), np.float64(0.3014360814914377))
Chord Played: (np.str_('Am'), np.float64(0.30816499402658115))
Chord Played: (np.str_('E'), np.float64(0.32121367476659085))
